In [1]:
import numpy as np
from rpy2.robjects import r, conversion, pandas2ri
from helper_functions import normalize_household_data, dict_to_named_list

In [2]:
pandas2ri.activate()
r.source('Simulator/Simulator.R')
model_r = r['simulate_and_reformat']

R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


R[write to console]: 
Attaching package: ‘actuar’


R[write to console]: The following objects are masked from ‘package:stats’:

    sd, var


R[write to console]: The following object is masked from ‘package:grDevices’:

    cm




In [3]:
def prior(batch_size: int) -> np.ndarray:
    param_batch = np.zeros((batch_size, 12))
    # alpha
    param_batch[:, 0] = np.random.uniform(0, 0.1, batch_size)
    # beta
    param_batch[:, 1] = np.random.uniform(0, 10, batch_size)
    # delta
    param_batch[:, 2] = np.random.uniform(-3, 3, batch_size)
    # mu_inf
    param_batch[:, 3:8] = np.exp(np.random.normal(0, 1, (batch_size, 5)))
    # mu_susc
    param_batch[:, 8:10] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    # mu_protect
    param_batch[:, 10:] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    return param_batch

test = prior(1).flatten()

In [4]:
param_names = ['alpha', 'beta', 'delta', 
               'mu_inf_SC', 'mu_inf_SA', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA',
               'mu_susc_C', 'mu_susc_A', 
               'mu_protect_acq', 'mu_protect_transm']

In [5]:
def simulator(params: np.ndarray,
              variant: str = "alpha",
              selection_procedure: str = "pedcov") -> np.ndarray:
    """
    Simulate data with given parameters and reformat it to a numpy array.
    :param params: parameters for the simulation
    :param variant: variant of the virus (alpha or omicron)
    :param selection_procedure: selection procedure for the households (pedcov or random)
    :return: simulated data as numpy array
    """
    # create dict from params and param_names
    par_dict = dict(zip(param_names, params))
    # update dict with fixed parameters
    par_dict.update({'variant': variant, 'selection_procedure': selection_procedure})

    # minimal_length should be the maximal length of the time series in the real data set
    if par_dict['variant'] == "alpha":
        minimal_length = 8
    elif par_dict['variant'] == "omicron":
        minimal_length = 9
    else:
        raise ValueError("Variant not supported. Must be 'alpha' or 'omicron'.")

    # simulate data
    sim_data_r = model_r(dict_to_named_list(par_dict))
    # convert to pandas dataframe
    sim_data_full = conversion.rpy2py(sim_data_r)
    # normalize data and return as numpy array
    sim_data_norm = normalize_household_data(sim_data_full, minimal_length=minimal_length, return_list=True)
    return sim_data_norm
#model_py.__name__ = 'simulate_and_reformat'

In [6]:
sim_data = simulator(test, selection_procedure='random')

In [7]:
test

array([ 0.0631339 ,  8.25501715, -1.59125124,  3.19537283,  0.6199249 ,
        0.19533058,  8.32441027,  5.54635151,  3.0737464 ,  2.21558969,
        1.66048763,  1.50983451])

In [10]:
print(sim_data[2])

[[ 0.          0.          0.          0.        ]
 [ 0.19620253  0.          0.          0.        ]
 [ 0.20886076  0.          1.          0.        ]
 [ 0.22151899  0.          0.          0.        ]
 [ 0.23417722  1.         -1.          0.        ]
 [ 0.23417722  1.          1.          0.        ]
 [ 0.23417722  0.          1.          0.        ]
 [ 0.87974684  0.          0.          0.        ]]


In [ ]:
# columns: date_sympt_norm, infect_status_norm, age_norm, protected
# rows: empty individuals in the beginning (households are of same size)
# last row: end_followup_norm with 0  # todo: maybe change to 1